<a href="https://colab.research.google.com/github/BarrenWuffet402/Ragify/blob/main/Ragify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Private Document Q&A (RAG)

This notebook builds a privacy-first RAG assistant for personal documents (PDF/TXT/MD).

In [1]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 6.8 MB/s eta 0:00:00


In [2]:
import re
import torch
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files

print('GPU available:', torch.cuda.is_available())

GPU available: False


In [3]:
uploaded = files.upload()  # Upload .pdf, .txt, .md files
print('Uploaded:', list(uploaded.keys()))

Saving TECH16_Day1_ClassNotes.html to TECH16_Day1_ClassNotes.html
Saving TECH16_Day2_ClassNotes.html to TECH16_Day2_ClassNotes.html
Uploaded: ['TECH16_Day1_ClassNotes.html', 'TECH16_Day2_ClassNotes.html']


In [6]:
def read_file_text(path: str) -> str:
    ext = path.lower().split('.')[-1]
    if ext == 'pdf':
        reader = PdfReader(path)
        text = '\n'.join((p.extract_text() or '') for p in reader.pages)
        return text
    if ext in ['txt', 'md']:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    return ''

def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150):
    text = re.sub(r'\s+', ' ', text).strip()
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

documents = []
for fn in uploaded.keys():
    txt = read_file_text(fn)
    if txt.strip():
        documents.append({'source': fn, 'text': txt})

chunks = []
for doc in documents:
    for c in chunk_text(doc['text']):
        chunks.append({'source': doc['source'], 'text': c})

print('Loaded docs:', [d['source'] for d in documents])
print('Total chunks:', len(chunks))

Loaded docs: []
Total chunks: 0


In [ ]:
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

chunk_texts = [c['text'] for c in chunks]
if not chunk_texts:
    raise ValueError('No text extracted. Upload at least one readable PDF/TXT/MD file.')

embeddings = embed_model.encode(chunk_texts, convert_to_numpy=True, normalize_embeddings=True)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity when embeddings are normalized
index.add(np.asarray(embeddings, dtype=np.float32))

print('FAISS index size:', index.ntotal)

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # If OOM, switch to Qwen/Qwen2.5-0.5B-Instruct

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map='auto'
)

print('Model loaded:', MODEL_ID)

In [ ]:
def retrieve(query: str, k: int = 4):
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = index.search(np.asarray(q_emb, dtype=np.float32), k)
    results = []
    for score, i in zip(scores[0], idxs[0]):
        if i == -1:
            continue
        item = chunks[i]
        results.append({
            'rank': len(results) + 1,
            'score': float(score),
            'source': item['source'],
            'text': item['text']
        })
    return results

def answer_with_rag(question: str, k: int = 4, max_new_tokens: int = 250):
    hits = retrieve(question, k=k)
    context = '

'.join([f"[{h['rank']}] Source: {h['source']}
{h['text']}" for h in hits])

    prompt = f"""You are a helpful assistant for personal documents.
Answer only using the provided context.
If the answer is not in context, say: "I don't see that in the provided documents."

Question: {question}

Context:
{context}

Return:
1) A concise answer
2) Sources used like [1], [2]
"""

    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    gen_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return response, hits

def ask_question(question: str, k: int = 4):
    answer, hits = answer_with_rag(question, k=k)
    print('Q:', question)
    print('
A:', answer)
    print('
Retrieved chunks:')
    for h in hits:
        print(f"[{h['rank']}] {h['source']} (score={h['score']:.3f})")
    return answer, hits

In [ ]:
demo_questions = [
    'Please explain difference between embeddings, tokens, an vectors?',
    'Is an LLM a knowledge database or a skillset?',
    'Kindly elaborate on the RISEN framework, and when to use it?'
]

for q in demo_questions:
    print('
' + '=' * 80)
    ask_question(q, k=4)

## Assignment Write-Up Notes
- Goal: Ask natural-language questions over personal files (PDF/TXT/MD).
- Privacy approach: No external API calls to proprietary LLM endpoints.
- Method: Chunk documents -> embed with MiniLM -> retrieve with FAISS -> grounded answer generation.
- Limitation: Colab runtime is cloud-hosted; stricter privacy requires local runtime (e.g., laptop + Ollama).